# LoRA v2 — Hard Sample Mining + Mixed Fine-tuning
- Sample the hard data samples" and "Validate on hard data samples"
- Ran inference on full training set with v2 model — found 541 hard samples (17.4% of training data) that v2 gets wrong
- Hard samples have longer prompts and almost always include lecture text (96% vs 84%)
- Fine-tunes v2 further on a 50/50 mix of hard + easy samples (1082 total) to learn the hard cases without forgetting the easy ones
- Uses very low learning rate (5e-6, 6x lower than original v2) to avoid catastrophic forgetting
- 1 epoch only — quick ~1 hour training run
- Native image resolution at inference (proven best)

## Cell 0 - Install

In [3]:
!pip install -q "transformers==4.57.1" "peft==0.18.0" bitsandbytes accelerate "pandas==2.2.2" "pillow==11.3.0" tqdm

## Cell 1 - Download Data

In [4]:
import kagglehub
from pathlib import Path
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [5]:
COMP_DIR = Path(kagglehub.competition_download("pixels-to-predictions"))
print("Downloaded to:", COMP_DIR)

Downloaded to: /root/.cache/kagglehub/competitions/pixels-to-predictions


## Cell 2 - Imports and Config

In [6]:
import os, json, random, gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_DISABLED"] = "true"
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModelForVision2Seq, get_cosine_schedule_with_warmup
from peft import PeftModel

SEED = 42; random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
CHOICE_LETTERS = "ABCDEFGHIJ"
BATCH_SIZE_INFER = 4

from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Cell 3 - Load Data

In [7]:
DATA_DIR = COMP_DIR / "data"
if not DATA_DIR.exists(): DATA_DIR = COMP_DIR

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

example_name = Path(val_df.iloc[0]["image_path"]).name
matches = list(COMP_DIR.rglob(example_name))
DATA_ROOT = matches[0].parents[2]
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 3109 | Val: 1048 | Test: 1008


## Cell 4 - Prompt + Image helpers

In [8]:
def clean_text(x):
    if pd.isna(x): return ""
    return str(x).strip()

def build_prompt(row, include_answer=False):
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))
    prompt = "<image>\nYou are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"
    if lecture: prompt += f"Lecture: {lecture}\n\n"
    if hint: prompt += f"Hint: {hint}\n\n"
    prompt += f"Question: {question}\n\nChoices:\n{choices_text}\n\nAnswer:"
    if include_answer:
        ans = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[ans]}. {choices[ans]}"
    return prompt

def load_image(row):
    path = DATA_ROOT / row["image_path"]
    return Image.open(path).convert("RGB")  # native resolution

def get_candidate_token_ids(tokenizer):
    cids = {}
    for letter in CHOICE_LETTERS:
        forms = [letter, " "+letter, "\n"+letter, letter+".", " "+letter+"."]
        ids = set()
        for f in forms:
            enc = tokenizer.encode(f, add_special_tokens=False)
            if enc: ids.add(enc[-1])
        cids[letter] = sorted(ids)
    return cids

## Cell 5 - Load v2

In [9]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, attn_implementation="eager",
)
model = PeftModel.from_pretrained(base_model, "/content/drive/MyDrive/lora_v2_best")
model.eval()
candidate_ids = get_candidate_token_ids(processor.tokenizer)
print("Loaded v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loaded v2


## Cell 6 - Inference function

In [10]:
def predict_batch(df_batch):
    images = [load_image(row) for _, row in df_batch.iterrows()]
    prompts = [build_prompt(row) for _, row in df_batch.iterrows()]
    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}
    with torch.inference_mode():
        logits = model(**inputs).logits[:, -1, :]
    log_probs = torch.log_softmax(logits, dim=-1)
    preds = []
    for i, (_, row) in enumerate(df_batch.iterrows()):
        scores = [log_probs[i, candidate_ids[CHOICE_LETTERS[ci]]].max().item()
                  for ci in range(len(row["choices"]))]
        preds.append(int(np.argmax(scores)))
    return preds

## Cell 7 - Find hard samples + save to Drive

In [11]:
model.eval()
train_preds = []
for start in tqdm(range(0, len(train_df), BATCH_SIZE_INFER), desc="Scoring train set"):
    batch = train_df.iloc[start:start+BATCH_SIZE_INFER]
    p = predict_batch(batch)
    train_preds.extend(p)
    torch.cuda.empty_cache()

train_df["pred"] = train_preds
train_df["correct"] = train_df["pred"] == train_df["answer"]

print(f"Train accuracy: {train_df['correct'].sum()}/{len(train_df)} = {train_df['correct'].mean():.4f}")
print(f"Hard samples: {(~train_df['correct']).sum()}")

hard_df = train_df[~train_df["correct"]].copy()
easy_df = train_df[train_df["correct"]].copy()

# Save to Drive so we don't lose it
hard_df.to_csv("/content/drive/MyDrive/hard_samples.csv", index=False)
print(f"Saved {len(hard_df)} hard samples to Drive")

Scoring train set:   0%|          | 0/778 [00:00<?, ?it/s]

Train accuracy: 2568/3109 = 0.8260
Hard samples: 541
Saved 541 hard samples to Drive


## Cell 8 - Prepare mixed dataset + train

In [13]:
mixed_df = pd.concat([
    hard_df,
    easy_df.sample(n=len(hard_df), random_state=SEED)
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Hard: {len(hard_df)} | Easy sampled: {len(hard_df)} | Mixed: {len(mixed_df)}")

class VQATrainDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = load_image(row)
        full_text = build_prompt(row, include_answer=True)
        full_inputs = processor(text=[full_text], images=[image], return_tensors="pt", truncation=False)
        answer_str = f" {CHOICE_LETTERS[int(row['answer'])]}. {row['choices'][int(row['answer'])]}"
        n_answer = len(processor.tokenizer.encode(answer_str, add_special_tokens=False))
        return {
            "input_ids": full_inputs["input_ids"].squeeze(0),
            "attention_mask": full_inputs["attention_mask"].squeeze(0),
            "pixel_values": full_inputs["pixel_values"].squeeze(0),
            "n_answer_tokens": n_answer,
        }

def collate_train(batch):
    max_len = max(x["input_ids"].shape[0] for x in batch)
    pad_id = processor.tokenizer.pad_token_id
    input_ids_list, attention_mask_list, labels_list, pixel_values_list = [], [], [], []
    for x in batch:
        seq_len = x["input_ids"].shape[0]
        pad_len = max_len - seq_len
        n_answer = x["n_answer_tokens"]
        input_ids = torch.cat([torch.full((pad_len,), pad_id, dtype=x["input_ids"].dtype), x["input_ids"]])
        attention_mask = torch.cat([torch.zeros(pad_len, dtype=x["attention_mask"].dtype), x["attention_mask"]])
        labels = torch.full_like(input_ids, -100)
        labels[max_len - n_answer:max_len] = input_ids[max_len - n_answer:max_len]
        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)
        pixel_values_list.append(x["pixel_values"])
    return {
        "input_ids": torch.stack(input_ids_list),
        "attention_mask": torch.stack(attention_mask_list),
        "labels": torch.stack(labels_list),
        "pixel_values": torch.stack(pixel_values_list),
    }

# Training
processor.tokenizer.padding_side = "right"
model.train()
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False

hard_ds = VQATrainDataset(mixed_df)
hard_loader = DataLoader(hard_ds, batch_size=1, shuffle=True,
                          collate_fn=collate_train, num_workers=2, pin_memory=True)

HARD_LR = 5e-6
HARD_EPOCHS = 1
GRAD_ACCUM = 16

total_steps = len(hard_loader)
opt_steps = total_steps // GRAD_ACCUM

for name, param in model.named_parameters():
    if "lora" in name.lower():
        param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,}")

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=HARD_LR, weight_decay=0.01
)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=max(1, int(opt_steps * 0.05)),
    num_training_steps=opt_steps,
)

loss_history = []
optimizer.zero_grad()

pbar = tqdm(hard_loader, desc="Hard+Easy mix (1 epoch)")
for step, batch in enumerate(pbar):
    batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}
    out = model(**batch)
    loss = out.loss / GRAD_ACCUM
    loss.backward()
    if (step + 1) % GRAD_ACCUM == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        optimizer.zero_grad(set_to_none=True)
    loss_history.append(out.loss.item())
    if (step + 1) % 50 == 0:
        pbar.set_postfix(loss=f"{loss_history[-1]:.4f}",
                         avg50=f"{np.mean(loss_history[-50:]):.4f}")

model.eval(); model.config.use_cache = True
torch.cuda.empty_cache()
print(f"\nDone! First 50 avg: {np.mean(loss_history[:50]):.4f}, Last 50 avg: {np.mean(loss_history[-50:]):.4f}")

import shutil
SAVE_DIR = Path("/content/lora_v2_hard")
DRIVE_SAVE = Path("/content/drive/MyDrive/lora_v2_hard")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
DRIVE_SAVE.mkdir(parents=True, exist_ok=True)
if DRIVE_SAVE.exists(): shutil.rmtree(DRIVE_SAVE)
shutil.copytree(SAVE_DIR, DRIVE_SAVE)
print(f"Saved to {DRIVE_SAVE}")

Hard: 541 | Easy sampled: 541 | Mixed: 1082
Trainable: 4,161,536


Hard+Easy mix (1 epoch):   0%|          | 0/1082 [00:00<?, ?it/s]


Done! First 50 avg: 0.1666, Last 50 avg: 0.1005
Saved to /content/drive/MyDrive/lora_v2_hard


## Cell 9 - Validate

In [14]:
model.eval()
gc.collect(); torch.cuda.empty_cache()
processor.tokenizer.padding_side = "left"

eval_df = val_df.copy().reset_index(drop=True)
all_preds = []
for start in tqdm(range(0, len(eval_df), BATCH_SIZE_INFER), desc="Val"):
    batch = eval_df.iloc[start:start+BATCH_SIZE_INFER]
    p = predict_batch(batch)
    all_preds.extend(p)
    torch.cuda.empty_cache()

acc = sum(p == a for p, a in zip(all_preds, eval_df["answer"])) / len(eval_df)
print(f"\nFull val accuracy: {acc:.4f}")
print(f"Previous best leaderboard: 0.768")

Val:   0%|          | 0/262 [00:00<?, ?it/s]


Full val accuracy: 0.7204
Previous best leaderboard: 0.768


## Cell 10 - Submit

In [15]:
all_preds = []
for start in tqdm(range(0, len(test_df), BATCH_SIZE_INFER), desc="Test"):
    batch = test_df.iloc[start:start+BATCH_SIZE_INFER]
    p = predict_batch(batch)
    all_preds.extend(p)
    torch.cuda.empty_cache()

submission = pd.DataFrame({"id": test_df["id"], "answer": all_preds})
submission.to_csv("/content/submission.csv", index=False)
submission.to_csv("/content/drive/MyDrive/submission_v2_hard.csv", index=False)
print(f"Saved ({len(submission)} rows)")

from google.colab import files
files.download("/content/submission.csv")

Test:   0%|          | 0/252 [00:00<?, ?it/s]

Saved (1008 rows)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## New Approach

This runs 3 configs per sample (native+full, native+no_lecture, 224+full) and averages their log-prob scores. Takes 3x longer than single inference (1.5 hours for full val+test) but no training. Each config sees the question slightly differently, and averaging smooths out individual errors.

## Cell: Multi-prompt ensemble + confidence routing

In [17]:
del model, base_model; gc.collect(); torch.cuda.empty_cache()

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, attn_implementation="eager",
)
model = PeftModel.from_pretrained(base_model, "/content/drive/MyDrive/lora_v2_best")
model.eval()
processor.tokenizer.padding_side = "left"
print("Loaded v2 (original best)")

Loaded v2 (original best)


In [20]:
model.eval()
gc.collect(); torch.cuda.empty_cache()

# Reload v2 (not the hard-tuned version)
del model, base_model; gc.collect(); torch.cuda.empty_cache()

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, attn_implementation="eager",
)
model = PeftModel.from_pretrained(base_model, "/content/drive/MyDrive/lora_v2_best")
model.eval()
processor.tokenizer.padding_side = "left"
print("Loaded v2 (original best)")

# Three prompt variants
def prompt_full(row):
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))
    prompt = "<image>\nYou are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"
    if lecture: prompt += f"Lecture: {lecture}\n\n"
    if hint: prompt += f"Hint: {hint}\n\n"
    prompt += f"Question: {question}\n\nChoices:\n{choices_text}\n\nAnswer:"
    return prompt

def prompt_no_lecture(row):
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    prompt = "<image>\nYou are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"
    if hint: prompt += f"Hint: {hint}\n\n"
    prompt += f"Question: {question}\n\nChoices:\n{choices_text}\n\nAnswer:"
    return prompt

def prompt_short(row):
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))
    prompt = f"<image>\nQuestion: {question}\n\nChoices:\n{choices_text}\n\nAnswer:"
    return prompt

# Image loaders
def img_native(row):
    return Image.open(DATA_ROOT / row["image_path"]).convert("RGB")

def img_224(row):
    return Image.open(DATA_ROOT / row["image_path"]).convert("RGB").resize((224,224), Image.BICUBIC)

def get_scores(df_batch, img_fn, prompt_fn):
    """Return raw log-prob scores for each choice."""
    images = [img_fn(row) for _, row in df_batch.iterrows()]
    prompts = [prompt_fn(row) for _, row in df_batch.iterrows()]
    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}
    with torch.inference_mode():
        logits = model(**inputs).logits[:, -1, :]
    log_probs = torch.log_softmax(logits, dim=-1)
    all_scores = []
    for i, (_, row) in enumerate(df_batch.iterrows()):
        scores = [log_probs[i, candidate_ids[CHOICE_LETTERS[ci]]].max().item()
                  for ci in range(len(row["choices"]))]
        all_scores.append(scores)
    return all_scores

def ensemble_predict(df_batch):
    """Average scores across multiple prompt+image configs, pick argmax."""
    configs = [
        (img_native, prompt_full),
        (img_native, prompt_no_lecture),
        (img_224, prompt_full),
    ]

    all_config_scores = []
    for img_fn, prompt_fn in configs:
        try:
            scores = get_scores(df_batch, img_fn, prompt_fn)
            all_config_scores.append(scores)
        except RuntimeError:
            torch.cuda.empty_cache()
            scores = []
            for j in range(len(df_batch)):
                s = get_scores(df_batch.iloc[j:j+1], img_fn, prompt_fn)
                scores.extend(s)
                torch.cuda.empty_cache()
            all_config_scores.append(scores)

    preds = []
    for i in range(len(df_batch)):
        n_choices = len(all_config_scores[0][i])
        avg_scores = []
        for ci in range(n_choices):
            vals = []
            for config_scores in all_config_scores:
                s = np.array(config_scores[i], dtype=np.float32)
                s = s - s.max()  # normalize per config
                vals.append(s[ci])
            avg_scores.append(np.mean(vals))
        preds.append(int(np.argmax(avg_scores)))
    return preds

Loaded v2 (original best)


## Cell: Validate ensemble on full val

In [21]:
eval_df = val_df.copy().reset_index(drop=True)
print(f"Full validation: {len(eval_df)} samples, 3-config ensemble")

ensemble_preds = []
for start in tqdm(range(0, len(eval_df), BATCH_SIZE_INFER), desc="Ensemble val"):
    batch = eval_df.iloc[start:start+BATCH_SIZE_INFER]
    p = ensemble_predict(batch)
    ensemble_preds.extend(p)
    torch.cuda.empty_cache()

ens_acc = sum(p == a for p, a in zip(ensemble_preds, eval_df["answer"])) / len(eval_df)
print(f"\nEnsemble val accuracy: {ens_acc:.4f}")
print(f"Previous v2 native val: 0.7156")
print(f"Previous best leaderboard: 0.768")

Full validation: 1048 samples, 3-config ensemble


Ensemble val:   0%|          | 0/262 [00:00<?, ?it/s]


Ensemble val accuracy: 0.7166
Previous v2 native val: 0.7156
Previous best leaderboard: 0.768


## Cell : Submit if improved

In [22]:
all_preds = []
for start in tqdm(range(0, len(test_df), BATCH_SIZE_INFER), desc="Ensemble test"):
    batch = test_df.iloc[start:start+BATCH_SIZE_INFER]
    p = ensemble_predict(batch)
    all_preds.extend(p)
    torch.cuda.empty_cache()

submission = pd.DataFrame({"id": test_df["id"], "answer": all_preds})
submission.to_csv("/content/submission.csv", index=False)
submission.to_csv("/content/drive/MyDrive/submission_ensemble.csv", index=False)
print(f"Saved ({len(submission)} rows)")

from google.colab import files
files.download("/content/submission.csv")

Ensemble test:   0%|          | 0/252 [00:00<?, ?it/s]

Saved (1008 rows)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>